# Neo4j docs examples

source : https://github.com/neo4j/neo4j-graphrag-python

In [ ]:
%pip install -r requirements.txt

In [15]:
from dotenv import load_dotenv
import os

if load_dotenv():
    print("Success: .env file found with some environment variables")
else:
    print("Caution: No environment variables found. Please create .env file in the root directory or add environment variables in the .env file")

Success: .env file found with some environment variables


## 1.Knowledge Graph Construction

Firs you have to set the neo4j service listening on 7687 port. I use the neo4j docker container

In [16]:
import asyncio

from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm import OpenAILLM

In [17]:
NEO4J_URI = os.getenv('DATABASE_URL')
NEO4J_USERNAME = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
OLLAMA_URL= os.getenv('OLLAMA_URL')

In [18]:
# Connect to the Neo4j database
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# List the entities and relations the LLM should look for in the text
node_types = ["Person", "House", "Planet"]
relationship_types = ["PARENT_OF", "HEIR_OF", "RULES"]
patterns = [
    ("Person", "PARENT_OF", "Person"),
    ("Person", "HEIR_OF", "House"),
    ("House", "RULES", "Planet"),
]

Embedder model

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

# Instantiate the embeddings model
embedder = HuggingFaceEmbeddings(
model_name="sentence-transformers/all-mpnet-base-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15222.53it/s]


In [13]:
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(model="phi3", temperature=0, base_url=OLLAMA_URL)

llm.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'model': 'phi3', 'created_at': '2026-05-02T15:30:14.99810192Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 80320808, 'load_duration': 11829888, 'prompt_eval_count': 10, 'prompt_eval_duration': 8201855, 'eval_count': 10, 'eval_duration': 58631668}, id='lc_run--019de94f-cf04-7e32-8ae6-608a06369424-0', tool_calls=[], invalid_tool_calls=[])

In [22]:
OLLAMA_URL

'http://localhost:11434'

In [ ]:
from neo4j_graphrag.llm import OllamaLLM
from neo4j_graphrag.embeddings.sentence_transformers import SentenceTransformerEmbeddings

llm = OllamaLLM(
    model_name="phi3"
    )

embedder = SentenceTransformerEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

TypeError: httpx.Client() got multiple values for keyword argument 'base_url'

In [14]:
# Instantiate the SimpleKGPipeline
kg_builder = SimpleKGPipeline(
    llm=llm,
    driver=driver,
    embedder=embedder,
    schema={
        "node_types": node_types,
        "relationship_types": relationship_types,
        "patterns": patterns,
    },
    on_error="IGNORE",
    from_file=False,
)

PipelineDefinitionError: 

In [ ]:


# Run the pipeline on a piece of text
text = (
    "The son of Duke Leto Atreides and the Lady Jessica, Paul is the heir of House "
    "Atreides, an aristocratic family that rules the planet Caladan."
)
asyncio.run(kg_builder.run_async(text=text))
driver.close()

PipelineDefinitionError: 